# Estimation des paramètres de l'onde cinématique

## Prérequis

Ce module utilise les informations des stations USGS sur le territoire continental des États-Unis (CONUS) pour construire un modèle statistique d'estimation des paramètres alpha et bêta de l'onde cinématique pour une grille donnée. Le modèle déjà entraîné est fourni ici.

Ce modèle nécessite des estimations moyennées par bassin des variables suivantes :
* Température moyenne annuelle (degrés C)
* Précipitation moyenne annuelle (pouces)
* Surface drainée (km²), fichier basin.area.tif issu de l'étape de moyenne par bassin


Ainsi que la variable calculée pixel par pixel :
* Rapport de relief (relief ratio)

### Remarques

* Selon votre version de Scikit-learn, il peut être nécessaire de réentraîner le modèle pour conserver la compatibilité (script EF5-KW-Modeling-master).
* La valeur nodata des rasters est supposée égale à -9999

In [3]:
from ef5_kw_estimate import ReadGrid, WriteGrid
import numpy as np
import pickle
from sklearn.impute import SimpleImputer
import rasterio


## Lecture des grilles d'entrée

In [4]:
# keepInfo=True conserve les informations de projection et de référence spatiale de cette grille...
# afin de les réutiliser lors de l'écriture des grilles de sortie
# Nous avons besoin du logarithme de la surface du bassin, calculé ici
basinArea = ReadGrid("input_grids/basin.area.tif",keepInfo=True)

basinArea = np.log10(basinArea)

temp     = ReadGrid("input_grids/mean_temp.avg.tif")
precip   = ReadGrid("input_grids/mean_precip.avg.tif")
precip = precip/12
rr       = ReadGrid("input_grids/relief.ratio.tif")


ERROR 1: PROJ: proj_create_from_database: datum not found
ERROR 1: PROJ: proj_create_from_database: datum not found
/var/folders/1z/2zs73sj958b3gtzwq06vr0280000gr/T/ipykernel_4064/3427228424.py:6: RuntimeWarning: invalid value encountered in log10
  basinArea = np.log10(basinArea)


In [5]:
print("Vérification des données raster...")
print("Valeurs maximales :", np.nanmax([basinArea, temp, precip, rr], axis=1))
print("Valeurs minimales :", np.nanmin([basinArea, temp, precip, rr], axis=1))

Checking raster data...
Max values: [  5.610286    29.254793   164.58334      0.19926934]
Min values: [-8.584308e-02 -9.999000e+03 -9.999000e+03 -9.999000e+03]


## Charger les modèles sérialisés (pickle) afin de les appliquer à nos données

In [6]:
scaler = pickle.load(open("scaler.p", "rb"), encoding='latin1')
alphamod = pickle.load(open("alpha_model.p", "rb"), encoding='latin1')
betamod = pickle.load(open("beta_model.p", "rb"), encoding='latin1')


/Users/vrobledodelgado/anaconda3/envs/vrobledod/lib/python3.8/site-packages/sklearn/base.py:310: UserWarning: Trying to unpickle estimator RobustScaler from version 1.5.2 when using version 0.24.2. This might lead to breaking code or invalid results. Use at your own risk.
  warnings.warn(
/Users/vrobledodelgado/anaconda3/envs/vrobledod/lib/python3.8/site-packages/sklearn/base.py:310: UserWarning: Trying to unpickle estimator SVR from version 1.5.2 when using version 0.24.2. This might lead to breaking code or invalid results. Use at your own risk.
  warnings.warn(


## Exécuter les modèles pour estimer alpha et bêta

In [7]:
print("Transformation vers l'espace des paramètres normalisé")
pred_real = scaler.transform(np.column_stack((basinArea, temp, precip, rr)))


Transforming to scaled parameter space


In [8]:
print("Calcul d'alpha")
alpha = alphamod.predict(pred_real)
alpha = np.power(10.0, alpha)
alpha.shape

Computing alpha


(1110510,)

In [9]:
print("Calcul de bêta")
beta = betamod.predict(pred_real)


Computing beta


In [10]:
# Le modèle extrapolant, on applique ici une astuce pour borner la valeur minimale de bêta à 0.
bad = np.where(beta < 0.0)
beta[bad] = np.exp(beta[bad] * 10.0)

with rasterio.open("input_grids/FAC_ghana1km.tif") as src: #Remplacez par la grille FAC correspondant à votre région
    fam = src.read(1)
# La grille FAC sert à masquer les zones qui ne conviennent pas au modèle
mask = fam > 7 #valeur minimale de la grille FAC pour cette résolution ; ce paramètre change selon la résolution des pixels
mask=np.reshape(mask,(1, -1))[:][0]
alpha[np.where(alpha>1000)]=np.nan
alpha[~mask] = np.nan
beta[~mask] = np.nan

## Écrire les nouvelles grilles de paramètres sur le disque

In [12]:
import osgeo.osr as osr
import osgeo.gdal as gdal
from osgeo.gdalconst import *
##si vous n'avez pas utilisé keepInfo dans le premier bloc, utilisez ceci.
def GetGridInfo(gridIn):
    dem = gdal.Open(gridIn, GA_ReadOnly)
    if dem is None:
        raise ValueError(f"Erreur lors de l'ouverture du fichier : {gridIn}")

    gt = dem.GetGeoTransform()
    proj = dem.GetProjection()
    nx = dem.RasterXSize
    ny = dem.RasterYSize
    
    print(f"Informations de grille extraites de {gridIn} : nx={nx}, ny={ny}")
    return gt, proj, nx, ny

gt, proj, nx, ny = GetGridInfo("input_grids/basin.area.tif")

def WriteGrid2(gridOutName, dataOut, gt, proj, nx, ny):
    driver = gdal.GetDriverByName('GTiff')
    dst_ds = driver.Create(gridOutName, nx, ny, 1, gdal.GDT_Float32)
    
    if dst_ds is None:
        raise RuntimeError("Échec de la création du fichier de sortie. Vérifiez les paramètres.")

    dst_ds.SetGeoTransform(gt)
    dst_ds.SetProjection(proj)
    dst_ds.GetRasterBand(1).WriteArray(dataOut.reshape(ny, nx))
    dst_ds.GetRasterBand(1).SetNoDataValue(-9999.0)
    dst_ds = None
    print(f"Écriture réussie de {gridOutName}")

# Appeler WriteGrid avec les métadonnées de `basinArea.tif`
WriteGrid2("./outputs/kw_alpha.tif", alpha, gt, proj, nx, ny)
WriteGrid2("./outputs/kw_beta.tif", beta, gt, proj, nx, ny)

Extracted Grid Info from input_grids/basin.area.tif: nx=914, ny=1215
Successfully wrote ./outputs/kw_alpha.tif
Successfully wrote ./outputs/kw_beta.tif


ERROR 1: PROJ: proj_create_from_database: datum not found
ERROR 1: PROJ: proj_create_from_database: datum not found


## Alpha0

Pour l'estimation d'alpha0, les entrées sont :
* Coefficient de rugosité de Manning
* Pente (en m/m) et accumulation d'écoulement.

La pente est calculée à partir du MNT, en supposant des coordonnées géographiques.

In [14]:
import rasterio
import numpy as np

# Définir une fonction de conversion des degrés en mètres
def degrees_to_meters(degrees_x, degrees_y, latitude):
    meters_per_degree_longitude = 111320 * np.cos(np.radians(latitude))
    meters_per_degree_latitude = 110574
    meters_x = degrees_x * meters_per_degree_longitude
    meters_y = degrees_y * meters_per_degree_latitude
    return meters_x, meters_y

# Lire le MNT
with rasterio.open("input_grids/DEM_ghana1km.tif") as src:
    dem = src.read(1)
    transform = src.transform
    profile = src.profile

# Calculer la taille de pixel en mètres
pixel_size_x = transform[0]
pixel_size_y = -transform[4]

# Calculer le gradient en mètres
height, width = dem.shape
lat_center = transform[5] + (height // 2) * transform[4]

res_x_m, res_y_m = degrees_to_meters(pixel_size_x, pixel_size_y, lat_center)

# Calculer le gradient
grad_y, grad_x = np.gradient(dem, res_y_m, res_x_m)

# Calculer la pente
slope = np.sqrt(grad_x**2 + grad_y**2)

# Écrire la pente dans un nouveau raster
out_meta = profile.copy()
out_meta.update({"dtype": "float32", "nodata": None})

with rasterio.open("outputs/slope_m_m.tif", "w", **out_meta) as dest:
    dest.write(slope.astype("float32"), 1)

In [ ]:
import numpy as np
import rasterio
from rasterio.transform import Affine
from rasterio.crs import CRS

# Lire le coefficient de Manning ; remplacez ce nom par la grille de coefficient de Manning de votre région
with rasterio.open("input_grids/manning_n_1km.tif") as src:
    manning = src.read(1)
    transform = src.transform
    crs = src.crs

# Lire la pente
with rasterio.open("outputs/slope_m_m.tif") as src:
    slope_grid = src.read(1)

# Lire l'accumulation d'écoulement
with rasterio.open("input_grids/FAC_ghana1km.tif") as src:
    facc = src.read(1)


# Calculer le coefficient de Chézy
COEM = 1.0 / manning
print(manning.shape)

# Calculer alpha0
alpha0 = np.zeros_like(facc, dtype=np.float32)
valid_mask = facc >= 0
print(facc.shape)
alpha0[valid_mask] = (1.0 / (COEM[valid_mask] * np.sqrt(slope_grid[valid_mask]))) ** (3 / 5)

# Borner les valeurs d'alpha0 à un maximum de 1000
max_valid = alpha0[alpha0 < 1000].max()
alpha0[alpha0 >= 1000] = max_valid

# Écrire alpha0 dans un nouveau raster
alpha0_grid = np.full_like(facc, np.nan, dtype=np.float32)
alpha0_grid[valid_mask] = alpha0[valid_mask]

output_meta = {
    'driver': 'GTiff',
    'height': alpha0_grid.shape[0],
    'width': alpha0_grid.shape[1],
    'count': 1,
    'dtype': 'float32',
    'crs': crs,
    'transform': transform
}

with rasterio.open('outputs/kw_alpha0.tif', 'w', **output_meta) as dst:
    dst.write(alpha0_grid, 1)

(1215, 914)
(1215, 914)


/var/folders/1z/2zs73sj958b3gtzwq06vr0280000gr/T/ipykernel_4064/2559801645.py:29: RuntimeWarning: divide by zero encountered in true_divide
  alpha0[valid_mask] = (1.0 / (COEM[valid_mask] * np.sqrt(slope_grid[valid_mask]))) ** (3 / 5)
